In [1]:
import pandas as pd
df = pd.read_csv('data.csv', encoding='ISO-8859-1')
print("Total rows found:", len(df))
df.head()

Total rows found: 541909


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [2]:
# 1. Remove rows where CustomerID is null (we can't track anonymous users)
df = df.dropna(subset=['CustomerID'])

# 2. Remove negative quantities (These are returns/cancellations)
# We only want successful sales for our first baseline model
df = df[df['Quantity'] > 0]

# 3. Remove rows with UnitPrice of 0 (Adjustments or freebies)
df = df[df['UnitPrice'] > 0]

# 4. Create the 'TotalSum' column (This is our Monetary value)
df['TotalSum'] = df['Quantity'] * df['UnitPrice']

# 5. Convert InvoiceDate to actual Date objects so Python understands time
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

print(f"Purge Complete! Rows remaining: {len(df)}")
df.head()

Purge Complete! Rows remaining: 397884


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalSum
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


In [16]:
import datetime as dt

# 1. Set 'snapshot date' to one day after the last invoice in the data
snapshot_date = df['InvoiceDate'].max() + dt.timedelta(days=1)

# 2. Group by CustomerID and calculate R, F, and M
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days, # Recency
    'InvoiceNo': 'count',                                   # Frequency
    'TotalSum': 'sum'                                       # Monetary
})

# 3. Rename columns for clarity
rfm.rename(columns={
    'InvoiceDate': 'Recency',
    'InvoiceNo': 'Frequency',
    'TotalSum': 'Monetary'
}, inplace=True)

print("RFM Table Created!")
rfm.head()

ETL Pipeline Fixed!
Final Columns: ['CustomerID', 'Recency', 'Frequency', 'Monetary']


,CustomerID,Recency,Frequency,Monetary
CustomerID,,,,
12346.0,12346,326,1,77183.60
12347.0,12347,2,182,4310.00
12348.0,12348,75,31,1797.24
12349.0,12349,19,73,1757.55
12350.0,12350,310,17,334.40


In [19]:
# 1. Create a fresh DataFrame using only the values, ignoring the problematic index
rfm_perfect = pd.DataFrame({
    'CustomerID': rfm.index.astype(int), # Forces the index to be the first column as an INT
    'Recency': rfm['Recency'].values,
    'Frequency': rfm['Frequency'].values,
    'Monetary': rfm['Monetary'].values
})

# 2. Export with index=False so the "index" isn't saved as a column
rfm_perfect.to_csv('rfm_data.csv', index=False)

print("Check the columns below - there should be exactly FOUR headers and NO duplicates.")
rfm_perfect.head()

Check the columns below - there should be exactly FOUR headers and NO duplicates.


,CustomerID,Recency,Frequency,Monetary
0,12346,326,1,77183.60
1,12347,2,182,4310.00
2,12348,75,31,1797.24
3,12349,19,73,1757.55
4,12350,310,17,334.40
